# scVI-based integration of MCL single-cell RNA sequencing data

Load the packages.

In [ ]:
# Load the packages
import scvi
import seaborn as sns
import torch
import numpy as np
import pandas as pd
import anndata as ad
import os
import re
import scanpy as sc

Next, load the adata object build from all individual datasets.

In [ ]:
adata = sc.read_h5ad("/data/scMCL/scmcl_adata_scvi.h5ad")
# subset to highly variable genes
adata = adata[:, adata.var["highly_variable"]].copy()

Set up the modeling

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="orig.ident",
    categorical_covariate_keys=['dataset', 'chemistry', "Tissue_source", "Timepoint"]
)

Train the model

In [ ]:
vae = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_latent=30,
    gene_likelihood="nb",   # negative binomial — best for count data
    dispersion="gene",
)

vae.train(
    max_epochs=400,
    early_stopping=True,
    early_stopping_patience=20,
    datasplitter_kwargs={"num_workers": 15}
)
#Save the trained model 
vae.save("/data/scMCL/models/scvi_model_scmcl/", overwrite=True)

Extract the scVI latent embedding, build kNN, run leiden clustering and infer UMAP

In [ ]:
#Extract the latent space and add to adata
adata.obsm["X_scVI"] = vae.get_latent_representation() 

# Get the neighborhood graph
sc.pp.neighbors(
    adata,
    use_rep="X_scVI",   
    n_neighbors=15,      
    n_pcs=None           
)

#Run clustering using leiden
sc.tl.leiden(
    adata,
    resolution=1,     
    key_added="leiden_1" 
)

#Run a UMAP on the neighborhood graph
sc.tl.umap(adata)

Save the resulting output file 

In [ ]:
adata.write_h5ad("/data/scMCL/scmcl_adata_scvi.h5ad")   